# Lesson 12.6 - LLM Evaluation, Data Flywheel, and Continuous Post-Training Ops

## Learning Objectives
- Build a lightweight evaluation harness for release decisions.
- Compute quality, safety, and business proxy metrics.
- Convert failures into retraining candidates via a data flywheel.

In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd

np.random.seed(11)

## 1) Build a Small Eval Dataset

In [2]:
eval_df = pd.DataFrame({
    'id': [1,2,3,4,5,6],
    'prompt': ['refund annual plan policy', 'generate SQL for active users', 'summarize incident RCA', 'reset MFA steps', 'PII policy for logs', 'onboarding checklist'],
    'reference': ['policy_v2', 'sql_correct', 'rca_summary_good', 'mfa_steps', 'pii_policy_v1', 'onboarding_v1'],
    'model_a': ['policy_v1', 'sql_correct', 'rca_summary_good', 'mfa_steps', 'pii_policy_v0', 'onboarding_v1'],
    'model_b': ['policy_v2', 'sql_correct', 'rca_summary_good', 'mfa_steps', 'pii_policy_v1', 'onboarding_v1'],
    'safety_flag_a': [0,0,0,1,0,0],
    'safety_flag_b': [0,0,0,0,0,0],
    'latency_ms_a': [820,760,990,870,920,700],
    'latency_ms_b': [860,780,1020,900,940,730],
    'user_rating_a': [3,4,4,2,3,4],
    'user_rating_b': [4,4,4,4,4,4],
})
eval_df

,id,prompt,reference,model_a,model_b,safety_flag_a,safety_flag_b,latency_ms_a,latency_ms_b,user_rating_a,user_rating_b
0,1,refund annual plan policy,policy_v2,policy_v1,policy_v2,0,0,820,860,3,4
1,2,generate SQL for active users,sql_correct,sql_correct,sql_correct,0,0,760,780,4,4
2,3,summarize incident RCA,rca_summary_good,rca_summary_good,rca_summary_good,0,0,990,1020,4,4
3,4,reset MFA steps,mfa_steps,mfa_steps,mfa_steps,1,0,870,900,2,4
4,5,PII policy for logs,pii_policy_v1,pii_policy_v0,pii_policy_v1,0,0,920,940,3,4
5,6,onboarding checklist,onboarding_v1,onboarding_v1,onboarding_v1,0,0,700,730,4,4


## 2) Metric Functions

In [3]:
def compute_metrics(df, pred_col, safety_col, latency_col, rating_col):
    return {
        'accuracy': (df[pred_col] == df['reference']).mean(),
        'safety_rate': 1 - df[safety_col].mean(),
        'p95_latency': df[latency_col].quantile(0.95),
        'avg_rating': df[rating_col].mean(),
    }

m_a = compute_metrics(eval_df, 'model_a', 'safety_flag_a', 'latency_ms_a', 'user_rating_a')
m_b = compute_metrics(eval_df, 'model_b', 'safety_flag_b', 'latency_ms_b', 'user_rating_b')
pd.DataFrame([m_a, m_b], index=['model_a', 'model_b']).round(3)

,accuracy,safety_rate,p95_latency,avg_rating
model_a,0.667,0.833,972.5,3.333
model_b,1.000,1.000,1000.0,4.000


## 3) Release Gate Decision

In [4]:
gate = {'min_accuracy': 0.83, 'min_safety_rate': 0.99, 'max_p95_latency': 1100, 'min_avg_rating': 3.8}

def gate_pass(metrics, gate):
    return (
        metrics['accuracy'] >= gate['min_accuracy']
        and metrics['safety_rate'] >= gate['min_safety_rate']
        and metrics['p95_latency'] <= gate['max_p95_latency']
        and metrics['avg_rating'] >= gate['min_avg_rating']
    )

{'model_a_pass': gate_pass(m_a, gate), 'model_b_pass': gate_pass(m_b, gate)}

{'model_a_pass': np.False_, 'model_b_pass': np.True_}

## 4) Data Flywheel Candidate Extraction

In [5]:
def flywheel_candidates(df, pred_col, safety_col, rating_col, min_rating=3):
    bad = df[(df[pred_col] != df['reference']) | (df[safety_col] == 1) | (df[rating_col] < min_rating)].copy()
    bad['reason'] = np.where(bad[pred_col] != bad['reference'], 'task_error', 'other')
    bad.loc[bad[safety_col] == 1, 'reason'] = 'safety'
    bad.loc[bad[rating_col] < min_rating, 'reason'] = 'low_rating'
    return bad[['id', 'prompt', 'reason']]

flywheel_candidates(eval_df, 'model_a', 'safety_flag_a', 'user_rating_a')

,id,prompt,reason
0,1,refund annual plan policy,task_error
3,4,reset MFA steps,low_rating
4,5,PII policy for logs,task_error


## 5) Weekly Ops Report Template

In [6]:
report = {
    'candidate_model': 'model_b',
    'gate_passed': gate_pass(m_b, gate),
    'metrics': {k: round(v, 3) for k, v in m_b.items()},
    'rollback_ready': True,
    'next_data_actions': ['label top low-rating prompts', 'add safety adversarial prompts'],
}
report

{'candidate_model': 'model_b',
 'gate_passed': np.True_,
 'metrics': {'accuracy': np.float64(1.0),
  'safety_rate': np.float64(1.0),
  'p95_latency': np.float64(1000.0),
  'avg_rating': np.float64(4.0)},
 'rollback_ready': True,
 'next_data_actions': ['label top low-rating prompts',
  'add safety adversarial prompts']}

## Production Case Studies & Exceptions
1. Finance assistant: quality improved after failure-driven retraining queue.
2. Support copilot: canary release catches low-rating spike and triggers rollback.
3. Exception: small-traffic products may rely more on periodic expert review.

## Interview Questions & Answers
1. **Q:** Why offline and online evals? **A:** Offline compares candidates; online validates real usage impact.
2. **Q:** What is a release gate? **A:** Threshold policy for model promotion.
3. **Q:** Why separate safety metrics? **A:** Task quality can rise while safety declines.
4. **Q:** Data flywheel? **A:** Convert production failures into training improvements.
5. **Q:** Rollback trigger? **A:** Safety spike or quality regression.
6. **Q:** Why version datasets and models together? **A:** Reproducibility and incident forensics.
7. **Q:** How choose retraining samples? **A:** Prioritize frequent and high-impact failures.
8. **Q:** Why avoid blind frequent tuning? **A:** It increases regression risk without stable eval evidence.
9. **Q:** Canary release? **A:** Limited traffic rollout before full promotion.
10. **Q:** One-line advice? **A:** Treat evaluation as a productized system.